# Phase 5a: Trend Detection

In [ ]:
import yaml
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath('../src'))
from trend import run_backtest_for_origin, generate_weight_combinations, calculate_trend_signals

posts = pd.read_parquet('../data/processed/posts_clean.parquet')
hashtags = pd.read_parquet('../data/processed/hashtags_clean.parquet')
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

backtest_origins = config['backtest_origins']
tune_origins = backtest_origins[:3]
confirm_origins = backtest_origins[3:]
a_values = [1e-3, 1e-4, 1e-5]
weight_combos = generate_weight_combinations()
pools = ['all', 'beauty', 'family', 'fashion', 'fitness', 'food', 'interior', 'pet', 'travel']

best_avg_corr = 0.07657981138507167
best_params = (0.001, (np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0)))

print(f"Best params: a={best_params[0]}, weights={best_params[1]} with corr={best_avg_corr:.4f}")

conf_corrs, freq_corrs, growth_corrs = [], [], []
for origin in confirm_origins:
    for pool in pools:
        res = run_backtest_for_origin(posts, hashtags, origin, config, pool, best_params[1], best_params[0])
        if res is not None:
            if not np.isnan(res['corr']): conf_corrs.append(res['corr'])
            if not np.isnan(res['corr_freq']): freq_corrs.append(res['corr_freq'])
            if not np.isnan(res['corr_growth']): growth_corrs.append(res['corr_growth'])

print(f"Confirm set avg corr (tuned): {np.mean(conf_corrs):.4f}")
print(f"Confirm set avg corr (freq baseline): {np.mean(freq_corrs):.4f}")
print(f"Confirm set avg corr (growth baseline): {np.mean(growth_corrs):.4f}")

theta = 0.10311637809762543
config['growth_smoothing'] = best_params[0]
config['weights_trend'] = list(best_params[1])
config['theta'] = theta
config['backtest_tune_fraction'] = 0.6
with open('../config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

reference_date = '2019-05-15'
final_trends = []
for pool in pools:
    ref_dt = pd.to_datetime(reference_date)
    signals = calculate_trend_signals(posts, hashtags, ref_dt, config, pool=pool)
    if not signals.empty: final_trends.append(signals)
        
final_df = pd.concat(final_trends, ignore_index=True)
final_df['run_id'] = 'run-v1'
out_cols = ['run_id', 'pool', 'hashtag', 'reference_date', 'frequency', 'growth', 'engagement', 'recency', 'trend_score', 'trend_direction']
final_df = final_df[out_cols]

os.makedirs('../artifacts/v1.0', exist_ok=True)
final_df.to_parquet('../artifacts/v1.0/trends.parquet')
print(f"Final trend row count: {len(final_df)}")
